## Chains + LangSmith

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" # Enable LangSmith tracing
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' # Set the LangSmith project name

In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model='gpt-5.1',
    temperature=1.5,
    base_url=BASE_URL,
    api_key=API_KEY
)

In [16]:
# Loading other dependencies

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser, JsonOutputParser

### Simple Chain

In [4]:
fact_template = ChatPromptTemplate.from_template(
    template='Generate 5 interesting facts about {topic} as short pointers'
)

prompt = fact_template.format(topic='space')
print(prompt)

Human: Generate 5 interesting facts about space as short pointers


In [46]:
result = llm.invoke(prompt)
print(result)

content='- There are more stars in the observable universe than grains of sand on all of Earth’s beaches.  \n- Neutron stars are so dense that a teaspoon of their material would weigh billions of tons on Earth.  \n- In space, metals like two pieces of the same type of metal can weld together on contact in a vacuum (cold welding).  \n- Venus rotates backward compared to most planets, so the Sun appears to rise in the west and set in the east there.  \n- A day on Mercury (one full rotation) is longer than its year (one orbit around the Sun).' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 18, 'total_tokens': 146, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 33, 'engine_ttlt_ms': 741, 'pre_inference_ms': 9

In [47]:
parser = StrOutputParser()
parsed_result = parser.invoke(result.content)
parsed_result

'- There are more stars in the observable universe than grains of sand on all of Earth’s beaches.  \n- Neutron stars are so dense that a teaspoon of their material would weigh billions of tons on Earth.  \n- In space, metals like two pieces of the same type of metal can weld together on contact in a vacuum (cold welding).  \n- Venus rotates backward compared to most planets, so the Sun appears to rise in the west and set in the east there.  \n- A day on Mercury (one full rotation) is longer than its year (one orbit around the Sun).'

In [48]:
# Creating simple chain

fact_chain = fact_template | llm | parser
result = fact_chain.invoke({'topic': 'ipl'})
print(result)

1. The IPL is the most-watched T20 cricket league in the world and has, at times, ranked among the top sports leagues globally by brand value.  
2. The first-ever IPL match (2008) saw Brendon McCullum score 158* for KKR, instantly redefining expectations for T20 batting.  
3. Chris Gayle holds the record for the highest individual IPL score: 175* off 66 balls for RCB in 2013.  
4. Lasith Malinga is among the top wicket-takers in IPL history, known for his deadly yorkers and effectiveness at the death overs.  
5. IPL introduced the “strategic timeout” concept in cricket, allowing teams mid-innings breaks to discuss tactics and adjust plans.


In [49]:
fact_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


### Sequential Chain  

Combining two or more simple chains

In [14]:
topic_template = ChatPromptTemplate.from_template(
    template = 'you give detailed information in a funny way on topic: {topic}'
)
str_parser = StrOutputParser()
topic_chain = topic_template | llm | str_parser

In [9]:
result = topic_chain.invoke({'topic': 'space'})
result

'Here are the most important *recent* developments and trends in space (roughly 2023–2026), organized by theme so you can skim or dive deeper where you like. If you tell me which areas you care about (rockets, Mars, black holes, etc.), I can expand on those specifically.\n\n---\n\n## 1. Rockets & Launch Industry\n\n### SpaceX: Starship and Falcon\n\n- **Starship test flights (2023–2025)**  \n  - Multiple full-stack test flights from Boca Chica, Texas.  \n  - Early flights: reached space but were lost before planned splashdown; focus on proving booster recovery, upper-stage reentry, and in-space engine restarts.  \n  - New Raptor engines, hot-staging ring (upper stage lights before separating), reinforced heat shield.  \n  - Goal: fully reusable super-heavy launch system with >100–150 tons to low Earth orbit (LEO), enabling cheaper large missions (Moon, Mars, massive satellites).\n\n- **Falcon 9/Falcon Heavy dominance**  \n  - Falcon 9 doing record launch cadence (many dozens per year).

In [10]:
tweet_template = ChatPromptTemplate.from_template(
    template = 'Being a good social media manager, I want a Tweet in questioning format to engage the audiance on the below content \n\n {content}'
)

tweet_chain = tweet_template | llm | str_parser
tweet_chain.invoke({'content': result})

'Which part of today’s space revolution blows your mind the most: reusable mega-rockets (like Starship), new Moon missions (Artemis, Chandrayaan, China’s plans), Mars & deep-space probes, exoplanets and signs of life, or mega-constellations changing Earth’s orbit—and why? 🚀🌓🔭'

In [11]:
seq_chain = topic_chain | tweet_chain

In [13]:
seq_chain.invoke({'topic': 'Late Nights Meetings'})

'“Late‑night meetings” are linked to poorer sleep, decision quality, and inclusion—yet many global teams still rely on them. In your org, are late‑night calls a necessary evil to span time zones, or a habit that should be replaced with async and fair‑time policies?'

In [17]:
# Sending 2 inputs 

topic_template = ChatPromptTemplate.from_template(
    """
    Generate:
    1. A funny explanation of the topic
    2. A catchy headline

    Return response in JSON format with keys:
    content
    headline

    Topic: {topic}
    """
)

json_parser = JsonOutputParser()

topic_chain = topic_template | llm | json_parser

In [18]:
topic_chain.invoke({'topic': 'Artificial Intelligence'})

{'content': 'Artificial Intelligence is what happens when computers stop just doing math homework and start trying to help with everything else in your life.\n\nImagine you hire an intern who:\n- reads faster than every human on Earth combined,\n- never sleeps,\n- but occasionally misunderstands you so badly it’s either terrifying or hilarious.\n\nYou say: “Recommend me a movie.”\nAI hears: “Please analyze my soul and expose all my secret guilty-pleasure genres.”\n\nUnder the hood, AI is basically:\n1. **Pattern-spotting on steroids** – It stares at absurd amounts of data until it figures out, for example, that cat photos usually contain fur, triangles (ears), and an attitude problem.\n2. **Fancy guesswork** – When it predicts the next word you’ll type, it’s not reading your mind; it’s just extremely good at, “People who wrote this sentence so far usually continue like *that*.”\n3. **Confidence without self-awareness** – It will give you a wrong answer with the unwavering confidence of

In [20]:
tweet_template = ChatPromptTemplate.from_template(
    """
    Create an engaging tweet in questioning style.

    Headline:
    {headline}

    Content:
    {content}
    """
)

tweet_chain = tweet_template | llm | str_parser

seq_chain = topic_chain | tweet_chain

In [21]:
seq_chain.invoke({'topic': 'Agentic AI'})

'Agentic AI: when your computer stops waiting for orders and starts acting like a superpowered intern ⚡  \n\nIf your old AI just answered questions, what happens when it starts:  \n– Planning your launch  \n– Writing the emails  \n– Queuing the social posts  \n– Running A/B tests  \n– Then calmly informing you *you’re* the bottleneck?\n\nAre you ready to manage an AI that proactively does the work instead of just talking about it?'

In [22]:
seq_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
  +------------------+     
  | JsonOutputParser |     
  +------------------+     
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *       

### Parallel Chains

In [34]:
from langchain_core.runnables import RunnableParallel

In [35]:
topic_template = ChatPromptTemplate.from_template(template = 'Give me a catcy title for topic: {topic}')
summary_template = ChatPromptTemplate.from_template(template = 'Summarize the topic: {topic} in a funny way')
tags_template = ChatPromptTemplate.from_template(template = 'Generate 5 relevant Hashtags for topic: {topic} to post on linkedIn')


topic_chain = topic_template | llm | str_parser
summary_chain = summary_template | llm | str_parser
tags_chain = tags_template | llm | str_parser

In [36]:
# Making the parallel chain
parallel_chain = RunnableParallel({
    "title": topic_chain,
    "summary": summary_chain,
    "tags": tags_chain
}) 

In [ ]:
result = parallel_chain.invoke({'topic': 'RAG in Finance'})

print(f"TITLE: {result['title']}\n")
print(f"SUMMARY: {result['summary']}\n")
print(f"HASHTAGS: {result['tags']}")

TITLE: 1. "RAGonomics: Redefining Financial Intelligence with Retrieval-Augmented Generation"  
2. "From Data Lakes to Smart Takes: RAG’s New Role in Finance"  
3. "RAG to Riches: Transforming Financial Insights with AI"  
4. "Retrieval-Augmented Finance: Turning Unstructured Data into Alpha"  
5. "The RAG Advantage: Next-Gen Decision Making on Wall Street"

SUMMARY: Imagine a bank where every time you ask a question, instead of one overworked analyst digging through PDFs, a small army of nerdy librarians sprints into an underground vault full of documents, grabs the right pages, and whispers the answers to a very smart parrot who then explains it to you in perfect English.

That’s RAG in finance.

RAG = Retrieval-Augmented Generation  
Translation: “Don’t let the AI make stuff up; make it read the docs first.”

In finance, that means:

1. **Less Confident Nonsense, More Boring Accuracy**  
   Without RAG, an AI might say:
   - “Yes, your mortgage rate is 0.5% and comes with a free yac

In [30]:
parallel_chain.get_graph().print_ascii()

                            +-----------------------------------+                              
                            | Parallel<topic,summary,tags>Input |                              
                            +-----------------------------------+                              
                          *******              *              *******                          
                     *****                     *                     *****                     
                 ****                          *                          ****                 
+--------------------+              +--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+              +--------------------+ 
           *                                   *                                   *           
           *                            

In [41]:
# Merging the Parallel chain with a seq chain

combine_template = ChatPromptTemplate.from_template(
    "Combine the following into a professional LinkedIn post:\n\n"
    "Title: {title}\n"
    "Summary: {summary}\n"
    "Tags: {tags}\n\n"
    "Make it engaging and add relevant emojis."
)

combine_chain = combine_template | llm | str_parser
final_chain = parallel_chain | combine_chain

In [42]:
result = final_chain.invoke({'topic': 'RAG in Annual Reports'})
print(result)

🚀 From Pages to Predictions: How RAG Turns Annual Reports into Living Insight  

Imagine you’re trying to read a 300‑page annual report.

- You: “What was revenue growth in Europe?”  
- Annual Report: “Forward‑looking statements… synergies… macroeconomic headwinds…”  
- You, 40 minutes later: “So… no?”  

That’s where Retrieval‑Augmented Generation (RAG) walks in like the nerdy superhero of corporate paperwork 🦸‍♂️📄  

---

### 💡 What is RAG in annual reports?

It’s basically an AI that:  
1️⃣ Grabs the exact info from the annual report (retrieval)  
2️⃣ Uses a “chatty brain” to explain it in plain language (generation)  
3️⃣ Tries very hard not to hallucinate numbers your CFO will scream at 📉😅  

---

### 📊 What it actually does

RAG turns vague questions into grounded answers with receipts:

- Turns “Where did our operating margin go?” into:  
  “On page 147, it says margins dropped from 18% to 15% because of higher input costs and your ambitious but financially questionable love of 

In [40]:
print(result.keys())

dict_keys(['topic', 'summary', 'tags'])


In [43]:
final_chain.get_graph().print_ascii()

                            +-----------------------------------+                              
                            | Parallel<title,summary,tags>Input |                              
                            +-----------------------------------+                              
                          *******              *              *******                          
                     *****                     *                     *****                     
                 ****                          *                          ****                 
+--------------------+              +--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+              +--------------------+ 
           *                                   *                                   *           
           *                            

### Conditional Chain

In [48]:
from langchain_core.runnables import RunnableBranch
from pydantic import BaseModel, Field
from typing import List, Literal

In [51]:
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative', 'neutral'] = Field(description="The sentiment of the feedback")

model = llm.with_structured_output(Feedback)

In [62]:
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that classifies the sentiment of feedback."),
    ("user", "What is the sentiment of the following feedback? {feedback}")
])

classificaition_chain = classification_prompt | model

In [63]:
result = classificaition_chain.invoke({'feedback': 'I love using LangSmith! It has improved my workflow significantly.'})
print(result)
print(type(result))

sentiment='positive'
<class '__main__.Feedback'>


In [55]:
# Defining the templates for the brances

positive_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    ("human","Generate a thank you note for this positive feedback: {feedback}."),
])

negative_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    ("human", "Generate an apology note and offer assistance for this negative feedback: {feedback}."),
])

neutral_feedback_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a request for more details for this neutral feedback: {feedback}."),
])

default_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a response that escalates this feedback to a human representative: {feedback}."),
])

In [64]:
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', positive_template | llm | str_parser),
    (lambda x: x.sentiment == 'negative', negative_template | llm | str_parser),
    (lambda x: x.sentiment == 'neutral', neutral_feedback_prompt | llm | str_parser),
    default_template | llm | str_parser  # Default template
)

chain = classificaition_chain | branch_chain

In [66]:
# Run the chain with an example review
# Good review - "The product is excellent. I really enjoyed using it and found it very helpful."
# Bad review - "The product is terrible. It broke after just one use and the quality is very poor."
# Neutral review - "The product is okay. It works as expected but nothing exceptional."
# Default - "I'm not sure about the product yet. Can you tell me more about its features and benefits?"

result = chain.invoke({"feedback": "I'm not sure about the product yet. Can you tell me more about its features and benefits?"})

print(result)

Thanks for your neutral feedback. To help us understand it better and make meaningful improvements, could you share a bit more detail on:

- What specifically worked well for you?
- What didn’t quite meet your expectations?
- Was there anything confusing, missing, or inconvenient?
- What is one thing we could change that would move your experience from “neutral” to “positive”?

Any additional context or examples you can provide (e.g., where you were stuck, what you were trying to accomplish) would be very helpful.


In [67]:
result = chain.invoke(
    {
        "feedback": "The product is okay. It works as expected but nothing exceptional."
    }
)

print(result)

Thanks for your neutral feedback. To help us understand your experience better and identify specific areas for improvement, could you share a bit more detail?

- What aspects did you find acceptable or just “okay,” rather than good or bad?
- Was there anything that felt confusing, inconvenient, or missing?
- Is there one change that would have made your experience clearly positive?

Any concrete examples or situations you can describe would be very helpful.


In [68]:
chain.get_graph().print_ascii()

    +-------------+    
    | PromptInput |    
    +-------------+    
           *           
           *           
           *           
+--------------------+ 
| ChatPromptTemplate | 
+--------------------+ 
           *           
           *           
           *           
    +------------+     
    | ChatOpenAI |     
    +------------+     
           *           
           *           
           *           
      +--------+       
      | Lambda |       
      +--------+       
           *           
           *           
           *           
      +--------+       
      | Branch |       
      +--------+       
           *           
           *           
           *           
   +--------------+    
   | BranchOutput |    
   +--------------+    


### Other runnables

#### Lambda 
For running own funcion

In [118]:
from langchain_core.runnables import RunnableLambda

uppercase = RunnableLambda(lambda x: x.upper())
print(uppercase.invoke('hellop'))

HELLOP


In [119]:
# def This is for the summantion
def add(a, b):
    return a*2+b*3

In [120]:
random_sum = RunnableLambda(lambda x: add(x['a'], x['b']))
random_sum.invoke({'a':2, 'b':3})

13

#### Passthorugh

Keep the original input while other runnables do extra work.


Don't do anything

In [82]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()

print(passthrough.invoke("hello"))

hello


In [ ]:
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative', 'neutral'] = Field(description="The sentiment of the feedback")

model = llm.with_structured_output(Feedback)

classification_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that classifies the sentiment of feedback."),
    ("user", "What is the sentiment of the following feedback? {feedback}")
])

classificaition_chain = classification_prompt | model

In [87]:
positive_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    ("human","Generate a thank you note for this positive feedback: {feedback}."),
])

negative_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    ("human", "Generate an apology note and offer assistance for this negative feedback: {feedback}."),
])

neutral_feedback_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a request for more details for this neutral feedback: {feedback}."),
])

default_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a response that escalates this feedback to a human representative: {feedback}."),
])


branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', positive_template | llm | str_parser),
    (lambda x: x.sentiment == 'negative', negative_template | llm | str_parser),
    (lambda x: x.sentiment == 'neutral', neutral_feedback_prompt | llm | str_parser),
    default_template | llm | str_parser  # Default template
)

In [89]:
print(classificaition_chain)
print(branch_chain)

first=ChatPromptTemplate(input_variables=['feedback'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that classifies the sentiment of feedback.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['feedback'], input_types={}, partial_variables={}, template='What is the sentiment of the following feedback? {feedback}'), additional_kwargs={})]) middle=[RunnableBinding(bound=ChatOpenAI(output_version=None, profile={'name': 'GPT-5.1', 'release_date': '2025-11-13', 'last_updated': '2025-11-13', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'struct

In [91]:
passthrough_chain = RunnablePassthrough()

chain = classificaition_chain | {
    'classificaion': passthrough_chain,
    'result': branch_chain
}

In [ ]:
chain.get_graph().print_ascii()


            +-------------+              
            | PromptInput |              
            +-------------+              
                    *                    
                    *                    
                    *                    
         +--------------------+          
         | ChatPromptTemplate |          
         +--------------------+          
                    *                    
                    *                    
                    *                    
             +------------+              
             | ChatOpenAI |              
             +------------+              
                    *                    
                    *                    
                    *                    
               +--------+                
               | Lambda |                
               +--------+                
                    *                    
                    *                    
                    *             

In [93]:
result = chain.invoke({"feedback": "The product is okay. It works as expected but nothing exceptional."})

print(result)

{'classificaion': Feedback(sentiment='neutral'), 'result': 'Thanks for sharing your neutral feedback. To better understand your experience and identify specific areas for improvement, could you provide a bit more detail on the following?\n\n- What aspects did you find acceptable or “just okay,” rather than positive or negative?\n- Were there any moments that stood out as particularly helpful or particularly frustrating?\n- Is there anything you expected that you didn’t get, or anything you got that you didn’t really need?\n- What one change would most improve your overall experience?\n\nAny additional context you can provide will help us make more meaningful improvements.'}
<class 'dict'>


In [95]:
print(type(result))
print(result.keys())
print(result['classificaion'])
print(type(result['classificaion']))
print(result['result'])

<class 'dict'>
dict_keys(['classificaion', 'result'])
sentiment='neutral'
<class '__main__.Feedback'>
Thanks for sharing your neutral feedback. To better understand your experience and identify specific areas for improvement, could you provide a bit more detail on the following?

- What aspects did you find acceptable or “just okay,” rather than positive or negative?
- Were there any moments that stood out as particularly helpful or particularly frustrating?
- Is there anything you expected that you didn’t get, or anything you got that you didn’t really need?
- What one change would most improve your overall experience?

Any additional context you can provide will help us make more meaningful improvements.


#### Runnign batch

In [98]:
seq_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
  +------------------+     
  | JsonOutputParser |     
  +------------------+     
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *       

In [99]:
result = seq_chain.batch([
    {'topic': 'RAG in Finance'},
    {'topic': 'AI in Quant and Assest Management'},
    {'topic': 'AI in Education'}
])

In [100]:
print(len(result))
print(type(result[1]))

3
<class 'langchain_core.messages.base.TextAccessor'>


In [101]:
print(result[1])

“Rise of the Robo-Quants: When Your Fund Manager Is a Supercomputer With Commitment Issues” 🤖📉  

If your “intern” never sleeps, drinks only electricity, builds models of your models, stress-tests 10,000 ways your portfolio can implode, and still gets blamed as “the black box” when it’s wrong…  

At what point do we admit our AI fund manager is just a hyper-intense robot guessing tomorrow’s opening bell with really good PR?


#### RunnableRetry
Becomes failproof, reties on failure

this is used with the model to retry

In [102]:
retry_model = ChatOpenAI(
    model='gpt-5.1',
    temperature=1.5,
    base_url=BASE_URL,
    api_key=API_KEY
).with_retry(
    stop_after_attempt=3
)

In [ ]:
retry_model.invoke('Ping Hi how are you')

AIMessage(content='Hi! I’m here and ready to help. How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 11, 'total_tokens': 38, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 33, 'engine_ttlt_ms': 199, 'pre_inference_ms': 100, 'service_tbt_ms': 5, 'service_ttft_ms': 712, 'service_ttlt_ms': 848, 'total_duration_ms': 759, 'user_visible_ttft_ms': 612}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DeIhMZXDMFmT7vu44VAd5W2f5WriK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e16b4-2ba6-7e33-b1b2-3e43f4aacd9a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11,

In [117]:
import random
import time
def fakey(x):
    random.seed(time.time())
    random_num = random.random()
    print(random_num)
    if random_num <0.6:
        print("FAILING")
        raise ValueError("Random failure")
    print('Passing')
    return x


fakeChain = RunnableLambda(fakey).with_retry(stop_after_attempt=5) | retry_model

fakeChain.batch(['Ping how are you', 'Ping how are you', 'Ping how are you'])


0.4037390222319357
FAILING
0.16743401580920014
FAILING
0.5520741708976759
FAILING
0.7298465763374978
Passing
0.925141176115112
Passing
0.6139826759842747
Passing


[AIMessage(content='I’m here and working. How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 10, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 34, 'engine_ttlt_ms': 147, 'pre_inference_ms': 92, 'service_tbt_ms': 11, 'service_ttft_ms': 687, 'service_ttlt_ms': 953, 'total_duration_ms': 854, 'user_visible_ttft_ms': 595}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DeIufbiSRKyFB95dzPCiGqP7MvJtc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e16c0-c267-77b2-b77a-d4bc7a4d0554-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_to